# 🗺️ Module 6 (Advanced): Star Schema with Station Location & Bydel

## Overview

In this notebook we enrich the station dimension with **Oslo bydel** (borough/district) information to enable geographic analysis of bike usage patterns.

**What you'll learn:**
- Create `dim_station_location` with a surrogate `location_key` and a `bydel` column
- Understand how to bring in a new lookup/reference dimension
- Query trips by bydel to find cross-district flow patterns

---

**Prerequisites:**
- Completed Module 3 (Gold Layer)
- `silver_trips` and `dim_station` tables exist


## 🎓 Concept: Geographic Dimension Enrichment

Oslo is divided into **15 bydeler** (urban districts / boroughs).  
Each bike station has GPS coordinates (`latitude`, `longitude`) but no bydel attribute in the raw data.

We solve this by:
1. Creating a **static bydel lookup** (embedded in the notebook as reference data)
2. Assigning each station to a bydel using coordinate ranges — a lightweight approach that works without external geo-APIs
3. Building `dim_station_location` with a fresh **surrogate key** (`location_key`) independent of `dim_station.station_key`

> 💡 In production you would join against an official geometry dataset (GeoJSON / PostGIS) using `ST_Within`. The bounding-box approach here is a pedagogical simplification.

### Oslo Bydel Map (simplified coordinate ranges)

| # | Bydel | Approx. lat range | Approx. lon range |
|---|-------|-------------------|-------------------|
| 1 | Gamle Oslo | 59.89 – 59.92 | 10.74 – 10.82 |
| 2 | Grünerløkka | 59.91 – 59.94 | 10.74 – 10.79 |
| 3 | Sagene | 59.93 – 59.96 | 10.73 – 10.78 |
| 4 | St. Hanshaugen | 59.92 – 59.95 | 10.71 – 10.75 |
| 5 | Frogner | 59.91 – 59.94 | 10.69 – 10.73 |
| 6 | Ullern | 59.90 – 59.93 | 10.63 – 10.70 |
| 7 | Vestre Aker | 59.93 – 59.97 | 10.65 – 10.73 |
| 8 | Nordre Aker | 59.95 – 59.99 | 10.71 – 10.79 |
| 9 | Bjerke | 59.93 – 59.97 | 10.79 – 10.86 |
| 10 | Grorud | 59.95 – 59.99 | 10.83 – 10.93 |
| 11 | Stovner | 59.97 – 60.02 | 10.88 – 10.98 |
| 12 | Alna | 59.90 – 59.95 | 10.83 – 10.95 |
| 13 | Østensjø | 59.86 – 59.91 | 10.79 – 10.89 |
| 14 | Nordstrand | 59.83 – 59.87 | 10.77 – 10.86 |
| 15 | Søndre Nordstrand | 59.81 – 59.85 | 10.74 – 10.82 |


## Step 1: Explore Station Coordinates

In [ ]:
%%sql
-- Preview station coordinates from dim_station
SELECT
    station_id,
    station_name,
    latitude,
    longitude
FROM dim_station
ORDER BY station_id
LIMIT 10

## Step 2: Create Bydel Reference Table

In [ ]:
%%sql
-- ============================================================
-- REFERENCE TABLE: ref_bydel
-- Static bounding-box lookup for Oslo's 15 bydeler
-- ============================================================
CREATE OR REPLACE TABLE ref_bydel (
    bydel_number    INT,
    bydel_name      STRING,
    lat_min         DOUBLE,
    lat_max         DOUBLE,
    lon_min         DOUBLE,
    lon_max         DOUBLE,
    area_type       STRING   -- inner_city / outer_west / outer_east / outer_south
) USING DELTA;

INSERT INTO ref_bydel VALUES
    ( 1, 'Gamle Oslo',          59.89, 59.92, 10.74, 10.82, 'inner_city'),
    ( 2, 'Grünerløkka',        59.91, 59.94, 10.74, 10.79, 'inner_city'),
    ( 3, 'Sagene',              59.93, 59.96, 10.73, 10.78, 'inner_city'),
    ( 4, 'St. Hanshaugen',      59.92, 59.95, 10.71, 10.75, 'inner_city'),
    ( 5, 'Frogner',             59.91, 59.94, 10.69, 10.73, 'inner_city'),
    ( 6, 'Ullern',              59.90, 59.93, 10.63, 10.70, 'outer_west'),
    ( 7, 'Vestre Aker',         59.93, 59.97, 10.65, 10.73, 'outer_west'),
    ( 8, 'Nordre Aker',         59.95, 59.99, 10.71, 10.79, 'outer_north'),
    ( 9, 'Bjerke',              59.93, 59.97, 10.79, 10.86, 'outer_east'),
    (10, 'Grorud',              59.95, 59.99, 10.83, 10.93, 'outer_east'),
    (11, 'Stovner',             59.97, 60.02, 10.88, 10.98, 'outer_east'),
    (12, 'Alna',                59.90, 59.95, 10.83, 10.95, 'outer_east'),
    (13, 'Østensjø',            59.86, 59.91, 10.79, 10.89, 'outer_south'),
    (14, 'Nordstrand',          59.83, 59.87, 10.77, 10.86, 'outer_south'),
    (15, 'Søndre Nordstrand',   59.81, 59.85, 10.74, 10.82, 'outer_south')

## Step 3: Create `dim_station_location`

In [ ]:
%%sql
-- ============================================================
-- DIMENSION TABLE: dim_station_location
-- Grain : one row per station (current state)
-- New surrogate key: location_key
-- ============================================================
CREATE OR REPLACE TABLE dim_station_location
USING DELTA
AS
SELECT
    -- New surrogate key independent of station_key in dim_station
    ROW_NUMBER() OVER (ORDER BY ds.station_id)  AS location_key,

    -- Business key (links back to dim_station and fact tables)
    ds.station_id,
    ds.station_name,
    ds.description,

    -- Geographic attributes
    ds.latitude,
    ds.longitude,

    -- Bydel enrichment
    COALESCE(rb.bydel_name,   'Unknown')        AS bydel,
    COALESCE(rb.bydel_number, 0)                AS bydel_number,
    COALESCE(rb.area_type,    'unknown')         AS area_type,

    -- Convenience flag
    CASE
        WHEN rb.area_type = 'inner_city' THEN TRUE
        ELSE FALSE
    END                                         AS is_inner_city

FROM dim_station ds
LEFT JOIN ref_bydel rb
    ON  ds.latitude  BETWEEN rb.lat_min AND rb.lat_max
    AND ds.longitude BETWEEN rb.lon_min AND rb.lon_max

## Step 4: Verify the Dimension

In [ ]:
%%sql
SELECT
    bydel,
    area_type,
    COUNT(*)    AS station_count
FROM dim_station_location
GROUP BY bydel, area_type
ORDER BY station_count DESC

In [ ]:
%%sql
-- How many stations have no bydel match?
SELECT COUNT(*) AS unmatched_stations
FROM dim_station_location
WHERE bydel = 'Unknown'

## Step 5: Trip Analysis by Bydel

In [ ]:
%%sql
-- Total trips by origin bydel
SELECT
    sl.bydel                        AS origin_bydel,
    sl.area_type,
    COUNT(*)                        AS total_trips,
    ROUND(AVG(st.duration / 60.0), 1) AS avg_duration_min
FROM silver_trips st
JOIN dim_station_location sl
    ON st.start_station_id = sl.station_id
WHERE st.is_valid = TRUE
GROUP BY sl.bydel, sl.area_type
ORDER BY total_trips DESC

In [ ]:
%%sql
-- Cross-bydel flow: which bydel pairs are most popular?
SELECT
    sl_start.bydel      AS from_bydel,
    sl_end.bydel        AS to_bydel,
    COUNT(*)            AS trips
FROM silver_trips st
JOIN dim_station_location sl_start ON st.start_station_id = sl_start.station_id
JOIN dim_station_location sl_end   ON st.end_station_id   = sl_end.station_id
WHERE st.is_valid = TRUE
  AND sl_start.bydel != sl_end.bydel    -- only cross-bydel trips
GROUP BY sl_start.bydel, sl_end.bydel
ORDER BY trips DESC
LIMIT 15

In [ ]:
%%sql
-- Inner city vs outer area trips
SELECT
    CASE WHEN sl.is_inner_city THEN 'Inner City' ELSE 'Outer Area' END AS origin_type,
    COUNT(*)                                AS trips,
    ROUND(AVG(st.duration / 60.0), 1)      AS avg_duration_min
FROM silver_trips st
JOIN dim_station_location sl ON st.start_station_id = sl.station_id
WHERE st.is_valid = TRUE
GROUP BY sl.is_inner_city
ORDER BY trips DESC

## 📌 Key Takeaways

- `dim_station_location` has its own **surrogate key** (`location_key`) separate from `dim_station.station_key`
- The **bydel** column is a classic example of enriching a dimension with reference data from an external lookup
- Bounding-box coordinate matching is simple and fast; use proper geometry functions (PostGIS, H3) in production
- Cross-bydel flow analysis opens up commuting pattern research
